In [1]:
# Imports
import sys
import os

# Add the upstream directory to sys.path
upstream_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if upstream_dir not in sys.path:
    sys.path.insert(0, upstream_dir)

# Now you can import the module
from opentrons_api import ot2_api
from microtissue_manipulator import core, utils, camera
import numpy as np 
import cv2
import time
import json
import keyboard
# from pynput import keyboard
import paths
import matplotlib.pyplot as plt
import requests
import datetime
import threading
import queue
import string
import pandas as pd
import multiprocessing as mp
from dataclasses import dataclass, fields, asdict, MISSING
from typing import get_type_hints, get_origin, get_args, Tuple, List, Dict, Any, Union
from ultralytics import YOLO
from enum import Enum

# Connect camera

In [2]:
cam_manager = camera.CameraManagerWindows()
print("Connected cameras:")
print(cam_manager.list_devices())

calibration_profile = 'noah_slice_config'

microscope_cam = camera.open_capture('microscope_cam', cam_manager=cam_manager)

Connected cameras:
['Arducam B0478 (USB3 48MP)', 'Digital Microscope', 'HD USB CAMERA']


In [3]:
microscope_cam.read()

(True,
 array([[[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]],
 
        [[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]],
 
        [[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]],
 
        ...,
 
        [[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]],
 
        [[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]],
 
        [[1, 0, 1],
         [1, 0, 1],
         [1, 0, 1],
         ...,
         [1, 0, 1],
         [1, 0, 1],
         [1, 0, 1]]], dtype=uint8))

# Init robot

In [4]:
# Connect the robot to the computer and this notebook
openapi = ot2_api.OpentronsAPI()

In [5]:
# Set the offset if the platform is used
openapi.add_slot_offsets([5,8,9], (0,0,64.2))

In [7]:
# Use the light control to see if the robot is connected as a sanity check
openapi.toggle_lights()

<Response [200]>

In [8]:
# Always do once after robot was just turned on
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [8]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

Total number of runs: 20
Current run ID: da3136e4-5f65-4425-a76b-eea3fd65817c
Current run status: idle


In [9]:
# Do after first launch
openapi.create_run()

Request status:
<Response [201]>
{
  "data": {
    "id": "3c45d3f3-d255-409d-8412-8f72453e2815",
    "ok": true,
    "createdAt": "2025-06-25T13:16:08.092567Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}


<Response [201]>

In [10]:
# Let the robot know that it has the P300 pipette
openapi.load_pipette()

Request status:
<Response [201]>
{
  "data": {
    "id": "8c884c67-1099-4491-8f59-65c5ab9f6c00",
    "createdAt": "2025-06-25T13:16:11.522902Z",
    "commandType": "loadPipette",
    "key": "8c884c67-1099-4491-8f59-65c5ab9f6c00",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "72e3406b-bdfd-4a55-8b1c-86ec4b7a1ccb"
    },
    "startedAt": "2025-06-25T13:16:11.526091Z",
    "completedAt": "2025-06-25T13:16:13.513512Z",
    "intent": "setup",
    "notes": []
  }
}


<Response [201]>

In [ ]:
openapi.move_labware(openapi.labware_dct['5'], 'offDeck')

# Labware declaration

### Opentrons 300 ul

In [11]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
r = openapi.load_labware(TIP_RACK, 11)

Labware ID:
54ade8a8-8dad-4e77-ac63-c496a20ec243



In [12]:
r = openapi.pick_up_tip(openapi.labware_dct['11'], "A2")

In [10]:
custom_labware_path = os.path.join(paths.BASE_DIR,'protocols','vwr_96_tiprack_200ul_xl.json')
with open(custom_labware_path, 'r') as json_file:
    custom_labware = json.load(json_file)

command_dict = {
            "data": custom_labware
        }

command_payload = json.dumps(command_dict)

url = openapi.get_url('runs')+ f'/{openapi.run_id}/'+ 'labware_definitions'
r = requests.post(url = url, headers = openapi.HEADERS, params = {"waitUntilComplete": True}, data = command_payload)
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "vwr_96_tiprack_200ul_xl"
#Load the tip rack. Slot = 1 by default.
openapi.load_labware(TIP_RACK, 10, namespace='custom_beta',verbose=True)

Labware ID:
85c7d9db-bc87-422b-af68-a0cc02b9eaa1



<Response [201]>

In [12]:
r = openapi.drop_tip_in_place()

In [11]:
r = openapi.pick_up_tip(openapi.labware_dct['10'], "F1")


### Well plate

In [13]:
WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
# WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 5, namespace='opentrons',verbose=True)

Offset (0, 0, 64.2) added to run for corning_96_wellplate_360ul_flat in slot 5.
Labware URI:
opentrons/corning_96_wellplate_360ul_flat/1

Check offset before using ...
Labware ID:
ac570414-d397-4eee-b93d-e04302821b12



# Calibration

In [14]:
calibration_profile = 'noah_slice_config'

calibration_data = utils.load_calibration_config(calibration_profile)
calib_origin = calibration_data['calib_origin']

spacing = 0.5  # Distance from the calib_point in mm
calibration_height = 96.5 # Z height for calibration points

# Calculate the four coordinates
calibration_points = [
    (calib_origin[0] + spacing, calib_origin[1] + spacing),  # Right 
    (calib_origin[0] + spacing, calib_origin[1] - spacing),  # Left
    (calib_origin[0] - spacing, calib_origin[1] - spacing),  # Up
    (calib_origin[0] - spacing, calib_origin[1] + spacing)   # Down
]

robot_coords = []
camera_coords = []

template = cv2.imread('../outputs/images/blade_template.png', 0)
template_height, template_width = template.shape[:2]

# window = cap.get_window()
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)

for calib_pt in calibration_points:
    openapi.move_to_coordinates((*calib_pt, calibration_height), min_z_height=1, verbose=False)
    cv2.waitKey(1000)
    # frame = cap.get_frame(undist=True)
    for _ in range(5):
        ret, frame = microscope_cam.read()
    frame = cv2.flip(frame, 1)
    # frame = frame_ops.undistort_frame(frame)
    
    x, y, z = openapi.get_position(verbose=False)[0].values()
    (text_width, text_height), _ = cv2.getTextSize(f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
    cv2.rectangle(frame, (10, 0), (10 + text_width, text_height + 70), (0, 0, 0), -1)
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    center_screen_x = frame.shape[1] // 2
    center_screen_y = frame.shape[0] // 2
    cv2.circle(frame, (center_screen_x, center_screen_y), 5, (0, 0, 255), -1)
    cv2.putText(frame, f"Center: ({center_screen_x}, {center_screen_y})", (center_screen_x + 10, center_screen_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Perform template matching
    result = cv2.matchTemplate(gray_frame, template, cv2.TM_CCOEFF_NORMED)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)


    # Eliminate overlapping matches
    locations = np.where(result >= 0.7)  # Adjust the threshold as needed
    rectangles = []
    for pt in zip(*locations[::-1]):
        rectangles.append([pt[0], pt[1], template_width, template_height])

    # Apply non-maximum suppression to remove overlapping rectangles
    rectangles, _ = cv2.groupRectangles(rectangles, groupThreshold=1, eps=0.2)
    for (x, y, w, h) in rectangles:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        center_x = x + w // 2
        center_y = y + h // 2
        cv2.circle(frame, (center_x, center_y), 3, (255, 0, 0), -1)
        calibration_point = (center_x+200, center_y)
        cv2.circle(frame, calibration_point, 4, (0, 255, 0), -1)
        cv2.putText(frame, f"({calibration_point[0]}, {calibration_point[1]})", (calibration_point[0] + 10, calibration_point[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        cv2.putText(frame, f"({center_x}, {center_y})", (center_x + 10, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv2.waitKey(1)
    cv2.imshow("video", frame)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    robot_coords.append((x, y))
    camera_coords.append(calibration_point)

cv2.destroyAllWindows()

calibration_data = utils.load_calibration_config(calibration_profile)

camera_coords = utils.sort_coordinates(camera_coords)
robot_coords = utils.sort_coordinates(robot_coords, reverse_y=True)

robot_to_camera_coords = {tuple(robot_coord): tuple(camera_coord) for robot_coord, camera_coord in zip(robot_coords, camera_coords)}
tf_mtx = utils.compute_tf_mtx(robot_to_camera_coords)

calibration_data['tf_mtx'] = tf_mtx.tolist()

utils.save_calibration_config(calibration_profile, calibration_data)

# Slice picking

In [18]:
openapi.toggle_lights()

<Response [200]>

In [9]:
openapi.get_position()

{'x': 155.88271357970297, 'y': 345.74300518592605, 'z': 151.79999999999998}


({'x': 155.88271357970297, 'y': 345.74300518592605, 'z': 151.79999999999998},
 <Response [201]>)

In [13]:
openapi.retract_axis('leftZ')
openapi.move_to_well(openapi.labware_dct['5'], 'A1', well_location='top', offset=(0,0,1))

<Response [201]>

In [16]:
openapi.retract_axis('leftZ')
openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

<Response [201]>

### Old version

In [ ]:
def on_mouse_click(event, cX, cY, flags, param):
    global X_init, Y_init, X, Y, target_x, target_y

    if event == cv2.EVENT_RBUTTONDOWN:
        x, y, _ = openapi.get_position(verbose=False)[0].values()
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

    if event == cv2.EVENT_MOUSEWHEEL:
        # List of well names in 96-well plate (A1-H12)
        if not hasattr(on_mouse_click, "well_index"):
            on_mouse_click.well_index = 0
        if not hasattr(on_mouse_click, "well_names"):
            rows = "ABCDEFGH"
            cols = range(1, 13)
            on_mouse_click.well_names = [f"{row}{col}" for row in rows for col in cols]
        num_wells = len(on_mouse_click.well_names)
        # event flags: positive for scroll up, negative for scroll down
        if flags > 0:
            on_mouse_click.well_index = (on_mouse_click.well_index + 1) % num_wells
        else:
            on_mouse_click.well_index = (on_mouse_click.well_index - 1) % num_wells
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        # print(f"Selected well: {current_well}")


# Create an instance of the ManualRobotMovement class
manual_movement = utils.ManualRobotMovement(openapi)
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)
# cv2.resizeWindow("video", 1050, 1348)
cv2.setMouseCallback("video", on_mouse_click)

target_x, target_y = 0, 0

dish_bottom = 70# - 11.5
pickup_offset = 0.0 #0.6
flow_rate = 28
volume = 10

x_asp, y_asp, z_asp = (300, 135, 110)

while True:
    ret, frame = microscope_cam.read()
    frame = cv2.flip(frame, 1)
    cv2.circle(frame, (target_x, target_y), 3, (0, 0, 255), -1)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Flow rate: {flow_rate} uL/s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Volume: {volume} uL", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    if hasattr(on_mouse_click, "well_names") and hasattr(on_mouse_click, "well_index"):
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        cv2.putText(frame, f"Current well: {current_well}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("video", frame)

    # Draw a point at the center of the screen
    key_pressed = cv2.waitKey(1)

    if key_pressed == ord('q'):
        keyboard.unhook_all()
        break

    elif key_pressed == ord('d'):
        openapi.dispense_in_place(flow_rate = flow_rate, volume = volume)

    elif key_pressed == ord('b'):
        openapi.blow_out_in_place(50)

    elif key_pressed == ord('a'):
        x_asp, y_asp, z_asp = openapi.get_position(verbose=False)[0].values()
        openapi.aspirate_in_place(flow_rate = flow_rate, volume = volume, verbose=True)

    elif key_pressed == ord('r'):
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((x_asp, y_asp, z_asp + 10), min_z_height=1, verbose=False)

    elif key_pressed == ord('g'):
        if current_well:
            openapi.retract_axis('leftZ')
            openapi.move_to_well(openapi.labware_dct['5'], well_name=current_well, well_location='top', offset = (-0.5, 0, 15), min_z_height=70, verbose=False)
# CREATE folder first, will not save photos unless folder already exists
    elif key_pressed == ord('s'):
        save_dir = "../outputs/images/2026-03-05_Frames_Calibration/"
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"frame_{timestamp}.png"
        filepath = os.path.join(save_dir, filename)
        cv2.imwrite(filepath, frame)
        print(f"Frame saved as {filepath}")
# Code directory to make new folder 

cv2.destroyAllWindows()

Saved position: {'x': 293.5608871761836, 'y': 133.86304089231612, 'z': 99.5}
Saved position: {'x': 295.5608871761836, 'y': 133.36304089231612, 'z': 101.0}
Saved position: {'x': 295.5608871761836, 'y': 133.46304089231612, 'z': 100.0}
Saved position: {'x': 295.5608871761836, 'y': 133.46304089231612, 'z': 99.0}
Saved position: {'x': 295.0608871761835, 'y': 133.46304089231612, 'z': 99.39999999999998}
Saved position: {'x': 294.56088717618337, 'y': 133.46304089231612, 'z': 99.59999999999997}
Saved position: {'x': 294.56088717618337, 'y': 133.46304089231612, 'z': 99.59999999999997}
Saved position: {'x': 294.56088717618337, 'y': 133.46304089231612, 'z': 99.59999999999997}
Saved position: {'x': 294.56088717618337, 'y': 133.46304089231612, 'z': 100.49999999999997}
Saved position: {'x': 154.67999999999995, 'y': 147.24, 'z': 76.42}
Saved position: {'x': 296.0774370498458, 'y': 133.208252372228, 'z': 102.0}
Saved position: {'x': 295.77743704984584, 'y': 133.208252372228, 'z': 101.0}
Saved position:

### Mouse control

In [16]:
def on_mouse_click(event, cX, cY, flags, param):
    global X_init, Y_init, X, Y, target_x, target_y

    if event == cv2.EVENT_RBUTTONDOWN:
        x, y, _ = openapi.get_position(verbose=False)[0].values()
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

    if event == cv2.EVENT_MOUSEWHEEL:
        # List of well names in 96-well plate (A1-H12)
        if not hasattr(on_mouse_click, "well_index"):
            on_mouse_click.well_index = 0
        if not hasattr(on_mouse_click, "well_names"):
            rows = "ABCDEFGH"
            cols = range(1, 13)
            on_mouse_click.well_names = [f"{row}{col}" for row in rows for col in cols]
        num_wells = len(on_mouse_click.well_names)
        # event flags: positive for scroll up, negative for scroll down
        if flags > 0:
            on_mouse_click.well_index = (on_mouse_click.well_index + 1) % num_wells
        else:
            on_mouse_click.well_index = (on_mouse_click.well_index - 1) % num_wells
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]

    if event == cv2.EVENT_LBUTTONDOWN:
        target_x, target_y = cX, cY
        cv2.circle(frame, (target_x, target_y), 5, (255, 0, 0), -1)
        X, Y, _ = tf_mtx @ (target_x, target_y, 1)

        _, _, curr_z = openapi.get_position(verbose=False)[0].values()
        openapi.move_to_coordinates((X, Y, curr_z), min_z_height=1, verbose=False)


calibration_data = utils.load_calibration_config(calibration_profile)
tf_mtx = np.array(calibration_data['tf_mtx'])

# Create an instance of the ManualRobotMovement class
manual_movement = utils.ManualRobotMovement(openapi)
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)
# cv2.resizeWindow("video", 1050, 1348)
cv2.setMouseCallback("video", on_mouse_click)

target_x, target_y = 0, 0

flow_rate = 280
volume = 10

x_asp, y_asp, z_asp = (300, 135, 110)

template = cv2.imread('../outputs/images/blade_template.png', 0)
template_height, template_width = template.shape[:2]

while True:
    ret, frame = microscope_cam.read()
    frame = cv2.flip(frame, 1)
    cv2.circle(frame, (target_x, target_y), 3, (0, 0, 255), -1)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Flow rate: {flow_rate} uL/s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Volume: {volume} uL", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    if hasattr(on_mouse_click, "well_names") and hasattr(on_mouse_click, "well_index"):
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        cv2.putText(frame, f"Current well: {current_well}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("video", frame)
    # Draw a point at the center of the screen
    key_pressed = cv2.waitKey(1)
    if key_pressed == ord('q'):
        keyboard.unhook_all()
        break

    elif key_pressed == ord('d'):
        openapi.dispense_in_place(flow_rate = flow_rate, volume = volume)

    elif key_pressed == ord('b'):
        openapi.blow_out_in_place(50)

    elif key_pressed == ord('a'):
        x_asp, y_asp, z_asp = openapi.get_position(verbose=False)[0].values()
        openapi.aspirate_in_place(flow_rate = flow_rate, volume = volume, verbose=True)

    elif key_pressed == ord('r'):
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((x_asp, y_asp, z_asp + 10), min_z_height=1, verbose=False)

    elif key_pressed == ord('g'):
        if current_well:
            openapi.retract_axis('leftZ')
            openapi.move_to_well(openapi.labware_dct['5'], well_name=current_well, well_location='top', offset = (-2.1, 0.6, 12), min_z_height=70, verbose=False)
# CREATE folder first, will not save photos unless folder already exists
    elif key_pressed == ord('s'):
        save_dir = "../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/"
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"frame_{timestamp}.png"
        filepath = os.path.join(save_dir, filename)
        cv2.imwrite(filepath, frame)
        print(f"Frame saved as {filepath}")
# Code directory to make new folder 

cv2.destroyAllWindows()

Frame saved as ../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/frame_20260401_145453.png
Frame saved as ../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/frame_20260401_145520.png
Frame saved as ../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/frame_20260401_145525.png
Frame saved as ../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/frame_20260401_145528.png
Frame saved as ../outputs/images/2026-03-31_PY75_250_frozen_agarose_tip3_1.2mmagarose/frame_20260401_145532.png
Request status:
<Response [201]>
{
  "data": {
    "id": "a0f50e06-d28b-411f-98bb-ba44536de49b",
    "createdAt": "2025-06-25T13:39:28.814597Z",
    "commandType": "aspirateInPlace",
    "key": "a0f50e06-d28b-411f-98bb-ba44536de49b",
    "status": "succeeded",
    "params": {
      "flowRate": 280.0,
      "volume": 10.0,
      "pipetteId": "72e3406b-bdfd-4a55-8b1c-86ec4b7a1ccb"
    },
    "result": {
      "volume": 10.0
    },
    "st